# Physical Health Risk Model Training

**Novelle — AI-Powered Maternal Health Risk Support Platform**

This notebook trains the physical/maternal health risk prediction model using an Ensemble approach.

## Model Overview
- **Target**: Physical health risk level (LOW / MEDIUM / HIGH)
- **Input Features**: BP, blood sugar, weight, symptoms, pregnancy week
- **Algorithm**: Ensemble (XGBoost + Random Forest + Logistic Regression)
- **Explainability**: SHAP values for feature importance

---

In [ ]:
# Install dependencies (run once)
# !pip install pandas numpy scikit-learn xgboost lightgbm shap imbalanced-learn matplotlib seaborn joblib

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

# ML imports
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, 
    f1_score, roc_auc_score, precision_score, recall_score,
    ConfusionMatrixDisplay
)
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import shap
import joblib

print("✅ Libraries loaded successfully")
print(f"   XGBoost version: {xgb.__version__}")

## 2. Load Data

In [ ]:
# Paths
DATA_DIR = Path('../datasets')
MODEL_DIR = Path('../../backend/app/ml/models')
REPORT_DIR = Path('../reports')

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Load datasets
health_df = pd.read_csv(DATA_DIR / 'synthetic_health_logs.csv')
profiles_df = pd.read_csv(DATA_DIR / 'synthetic_profiles.csv')

print(f"Health log records: {len(health_df):,}")
print(f"User profiles: {len(profiles_df):,}")
health_df.head()

## 3. Exploratory Data Analysis

In [ ]:
# Dataset info
print("=" * 50)
print("HEALTH LOG DATASET INFO")
print("=" * 50)
print(health_df.info())
print("\n" + "=" * 50)
print("STATISTICAL SUMMARY")
print("=" * 50)
health_df.describe()

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Risk label distribution
risk_counts = health_df['physical_risk_label'].value_counts()
colors = {'LOW': '#4CAF50', 'MEDIUM': '#FFC107', 'HIGH': '#F44336'}
risk_counts.plot(kind='bar', ax=axes[0], color=[colors.get(x, '#666') for x in risk_counts.index])
axes[0].set_title('Physical Health Risk Distribution')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# BP distribution by risk
for label in ['LOW', 'MEDIUM', 'HIGH']:
    if label in health_df['physical_risk_label'].values:
        subset = health_df[health_df['physical_risk_label'] == label]
        axes[1].scatter(subset['bp_systolic'], subset['bp_diastolic'], 
                       alpha=0.4, label=label, color=colors[label], s=10)
axes[1].set_title('Blood Pressure by Risk Level')
axes[1].set_xlabel('Systolic BP (mmHg)')
axes[1].set_ylabel('Diastolic BP (mmHg)')
axes[1].legend()
axes[1].axhline(y=90, color='red', linestyle='--', alpha=0.5, label='Hypertension threshold')
axes[1].axvline(x=140, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(REPORT_DIR / 'physical_health_eda.png', dpi=150)
plt.show()

In [ ]:
# Feature correlation heatmap
numeric_cols = ['bp_systolic', 'bp_diastolic', 'blood_sugar_fasting', 'blood_sugar_postmeal',
                'weight_kg', 'sleep_quality', 'pain_score', 'nausea_count', 'appetite_score']
plt.figure(figsize=(10, 8))
sns.heatmap(health_df[numeric_cols].corr(), annot=True, cmap='RdYlGn_r', center=0, fmt='.2f')
plt.title('Feature Correlation Matrix - Physical Health')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'physical_health_correlation.png', dpi=150)
plt.show()

## 4. Feature Engineering

In [ ]:
# Merge with profile data
df = health_df.merge(
    profiles_df[['user_id', 'age', 'bmi', 'hemoglobin_level', 'gestational_diabetes', 
                 'chronic_hypertension', 'previous_pregnancies']], 
    on='user_id', how='left'
)

# Feature engineering
df['bp_mean'] = (df['bp_systolic'] + df['bp_diastolic']) / 2
df['bp_pulse_pressure'] = df['bp_systolic'] - df['bp_diastolic']
df['blood_sugar_range'] = df['blood_sugar_postmeal'] - df['blood_sugar_fasting']
df['hypertension_flag'] = ((df['bp_systolic'] >= 140) | (df['bp_diastolic'] >= 90)).astype(int)
df['hyperglycemia_flag'] = ((df['blood_sugar_fasting'] >= 126) | (df['blood_sugar_postmeal'] >= 200)).astype(int)
df['symptom_count'] = df['dizziness'].astype(int) + df['edema_flag'].astype(int) + \
                      df['bleeding_flag'].astype(int) + df['cramps_flag'].astype(int)
df['overall_discomfort'] = df['pain_score'] + df['nausea_count'] * 2 + df['cramps_intensity']

# Trimester from pregnancy week
df['trimester'] = pd.cut(df['pregnancy_week'], bins=[0, 13, 27, 42], labels=[1, 2, 3]).astype(float)

# Convert boolean columns
bool_cols = ['dizziness', 'edema_flag', 'bleeding_flag', 'cramps_flag', 
             'gestational_diabetes', 'chronic_hypertension']
for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].astype(int)

print(f"Engineered features added. Total features: {len(df.columns)}")
df.head()

## 5. Prepare Training Data

In [ ]:
# Define features
FEATURE_COLS = [
    # Vital signs
    'bp_systolic', 'bp_diastolic', 'bp_mean', 'bp_pulse_pressure',
    'blood_sugar_fasting', 'blood_sugar_postmeal', 'blood_sugar_range',
    'weight_kg',
    # Symptoms
    'sleep_quality', 'pain_score', 'nausea_count', 'appetite_score',
    'dizziness', 'edema_flag', 'bleeding_flag', 'cramps_flag', 'cramps_intensity',
    'hydration_ml',
    # Engineered
    'hypertension_flag', 'hyperglycemia_flag', 'symptom_count', 'overall_discomfort',
    # Demographics
    'age', 'bmi', 'pregnancy_week', 'trimester', 'previous_pregnancies',
    'hemoglobin_level', 'gestational_diabetes', 'chronic_hypertension'
]

# Ensure all columns exist
available_features = [col for col in FEATURE_COLS if col in df.columns]
print(f"Using {len(available_features)} features out of {len(FEATURE_COLS)} defined")

# Prepare X and y
X = df[available_features].copy()
y = df['physical_risk_label'].copy()

# Handle missing values
X = X.fillna(X.median())

# Encode labels
label_encoder = LabelEncoder()
label_encoder.fit(['LOW', 'MEDIUM', 'HIGH'])
y_encoded = label_encoder.transform(y)

print(f"\nFeatures: {X.shape[1]}")
print(f"Samples: {X.shape[0]}")
print(f"\nClass distribution:")
for i, cls in enumerate(label_encoder.classes_):
    print(f"  {cls}: {(y_encoded == i).sum()} ({(y_encoded == i).mean()*100:.1f}%)")

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Train set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Handle class imbalance with SMOTE
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print(f"After SMOTE: {len(X_train_balanced)} samples")
for i, cls in enumerate(label_encoder.classes_):
    print(f"  {cls}: {(y_train_balanced == i).sum()}")

## 6. Model Training - Ensemble Approach

In [ ]:
# Individual models
print("Training individual models...\n")

# 1. XGBoost
xgb_clf = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

# 2. Random Forest
rf_clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

# 3. Logistic Regression
lr_clf = LogisticRegression(
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)

# Train individual models
xgb_clf.fit(X_train_balanced, y_train_balanced)
print("✅ XGBoost trained")

rf_clf.fit(X_train_balanced, y_train_balanced)
print("✅ Random Forest trained")

lr_clf.fit(X_train_balanced, y_train_balanced)
print("✅ Logistic Regression trained")

In [ ]:
# Evaluate individual models
models = {
    'XGBoost': xgb_clf,
    'Random Forest': rf_clf,
    'Logistic Regression': lr_clf
}

print("\n" + "=" * 60)
print("INDIVIDUAL MODEL PERFORMANCE")
print("=" * 60)

for name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    print(f"{name:25} | Accuracy: {acc:.4f} | F1: {f1:.4f}")

In [ ]:
# Create VotingClassifier ensemble
# Note: XGBoost in VotingClassifier can be tricky; we'll use soft voting with RF and LR
ensemble = VotingClassifier(
    estimators=[
        ('rf', rf_clf),
        ('lr', lr_clf)
    ],
    voting='soft',
    n_jobs=-1
)

# Train ensemble
print("\nTraining Voting Ensemble (RF + LR)...")
ensemble.fit(X_train_balanced, y_train_balanced)
print("✅ Ensemble trained")

In [ ]:
# Compare ensemble with XGBoost standalone
y_pred_ensemble = ensemble.predict(X_test_scaled)
y_pred_xgb = xgb_clf.predict(X_test_scaled)

acc_ensemble = accuracy_score(y_test, y_pred_ensemble)
acc_xgb = accuracy_score(y_test, y_pred_xgb)
f1_ensemble = f1_score(y_test, y_pred_ensemble, average='weighted')
f1_xgb = f1_score(y_test, y_pred_xgb, average='weighted')

print("\n" + "=" * 60)
print("ENSEMBLE vs XGBOOST COMPARISON")
print("=" * 60)
print(f"Ensemble (RF+LR)         | Accuracy: {acc_ensemble:.4f} | F1: {f1_ensemble:.4f}")
print(f"XGBoost Standalone       | Accuracy: {acc_xgb:.4f} | F1: {f1_xgb:.4f}")

# Use the better performing model
if f1_xgb > f1_ensemble:
    best_model = xgb_clf
    best_model_name = 'XGBoost'
    y_pred = y_pred_xgb
else:
    best_model = ensemble
    best_model_name = 'Ensemble'
    y_pred = y_pred_ensemble

print(f"\n→ Using {best_model_name} as final model")

## 7. Model Evaluation

In [ ]:
# Get probabilities from best model
y_pred_proba = best_model.predict_proba(X_test_scaled)

# Metrics
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

# AUC-ROC
try:
    auc_roc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='weighted')
except:
    auc_roc = 0.0

print("\n" + "=" * 50)
print("MODEL EVALUATION METRICS")
print("=" * 50)
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {f1:.4f}")
print(f"  AUC-ROC:   {auc_roc:.4f}")

In [ ]:
# Classification report
print("\n" + "=" * 50)
print("CLASSIFICATION REPORT")
print("=" * 50)
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)
disp.plot(ax=ax, cmap='Greens', values_format='d')
plt.title('Physical Health Risk - Confusion Matrix')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'physical_health_confusion_matrix.png', dpi=150)
plt.show()

## 8. Feature Importance & SHAP Analysis

In [ ]:
# Feature importance (use XGBoost for interpretability)
feature_importance = pd.DataFrame({
    'feature': available_features,
    'importance': xgb_clf.feature_importances_
}).sort_values('importance', ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(feature_importance['feature'], feature_importance['importance'], color='teal')
plt.xlabel('Importance')
plt.title('XGBoost Feature Importance - Physical Health Model')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'physical_health_feature_importance.png', dpi=150)
plt.show()

In [ ]:
# SHAP values (using XGBoost)
print("Computing SHAP values...")
explainer = shap.TreeExplainer(xgb_clf)
shap_values = explainer.shap_values(X_test_scaled[:100])

# SHAP summary plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test_scaled[:100], feature_names=available_features,
                  class_names=label_encoder.classes_, show=False)
plt.tight_layout()
plt.savefig(REPORT_DIR / 'physical_health_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Save Model Artifacts

In [ ]:
# Save ensemble model (or XGBoost if better)
# We save XGBoost as it's more commonly used and easier to explain
joblib.dump(xgb_clf, MODEL_DIR / 'physical_health_ensemble.joblib')
joblib.dump(scaler, MODEL_DIR / 'physical_health_scaler.joblib')
joblib.dump(label_encoder, MODEL_DIR / 'physical_health_label_encoder.joblib')

# Save feature columns for inference
with open(MODEL_DIR / 'physical_health_features.json', 'w') as f:
    json.dump(available_features, f)

print("✅ Model artifacts saved:")
print(f"   - {MODEL_DIR / 'physical_health_ensemble.joblib'}")
print(f"   - {MODEL_DIR / 'physical_health_scaler.joblib'}")
print(f"   - {MODEL_DIR / 'physical_health_label_encoder.joblib'}")
print(f"   - {MODEL_DIR / 'physical_health_features.json'}")

In [ ]:
# Save evaluation metrics
metrics = {
    'physical_health': {
        'model': best_model_name,
        'accuracy': round(accuracy, 4),
        'precision': round(precision, 4),
        'recall': round(recall, 4),
        'f1_score': round(f1, 4),
        'auc_roc': round(auc_roc, 4),
        'feature_columns': available_features
    }
}

# Load existing or create new
metrics_file = REPORT_DIR / 'evaluation_metrics.json'
if metrics_file.exists():
    with open(metrics_file, 'r') as f:
        all_metrics = json.load(f)
    all_metrics.update(metrics)
else:
    all_metrics = metrics

with open(metrics_file, 'w') as f:
    json.dump(all_metrics, f, indent=2)

print(f"\n✅ Metrics saved to {metrics_file}")

## 10. Model Inference Test

In [ ]:
def predict_physical_risk(bp_systolic, bp_diastolic, blood_sugar_fasting, blood_sugar_postmeal,
                          weight_kg, sleep_quality=3, pain_score=0, nausea_count=0,
                          dizziness=0, edema=0, bleeding=0, cramps=0, cramps_intensity=0,
                          appetite=3, hydration=1500, pregnancy_week=20,
                          age=28, bmi=24, hemoglobin=12, gestational_diabetes=0, 
                          chronic_hypertension=0, previous_pregnancies=0):
    """Predict physical health risk from input features."""
    # Load artifacts
    model = joblib.load(MODEL_DIR / 'physical_health_ensemble.joblib')
    scaler_loaded = joblib.load(MODEL_DIR / 'physical_health_scaler.joblib')
    encoder = joblib.load(MODEL_DIR / 'physical_health_label_encoder.joblib')
    
    # Engineered features
    bp_mean = (bp_systolic + bp_diastolic) / 2
    bp_pulse_pressure = bp_systolic - bp_diastolic
    blood_sugar_range = blood_sugar_postmeal - blood_sugar_fasting
    hypertension_flag = int(bp_systolic >= 140 or bp_diastolic >= 90)
    hyperglycemia_flag = int(blood_sugar_fasting >= 126 or blood_sugar_postmeal >= 200)
    symptom_count = dizziness + edema + bleeding + cramps
    overall_discomfort = pain_score + nausea_count * 2 + cramps_intensity
    trimester = 1 if pregnancy_week <= 13 else (2 if pregnancy_week <= 27 else 3)
    
    features = np.array([[
        bp_systolic, bp_diastolic, bp_mean, bp_pulse_pressure,
        blood_sugar_fasting, blood_sugar_postmeal, blood_sugar_range,
        weight_kg, sleep_quality, pain_score, nausea_count, appetite,
        dizziness, edema, bleeding, cramps, cramps_intensity, hydration,
        hypertension_flag, hyperglycemia_flag, symptom_count, overall_discomfort,
        age, bmi, pregnancy_week, trimester, previous_pregnancies,
        hemoglobin, gestational_diabetes, chronic_hypertension
    ]])
    
    features_scaled = scaler_loaded.transform(features)
    prediction = model.predict(features_scaled)[0]
    probabilities = model.predict_proba(features_scaled)[0]
    
    risk_label = encoder.inverse_transform([prediction])[0]
    confidence = probabilities[prediction]
    
    return {
        'risk_level': risk_label,
        'confidence': round(confidence, 3),
        'probabilities': {cls: round(prob, 3) for cls, prob in zip(encoder.classes_, probabilities)}
    }

# Test cases
print("\n" + "=" * 50)
print("INFERENCE TEST")
print("=" * 50)

# Low risk profile
result = predict_physical_risk(
    bp_systolic=115, bp_diastolic=75, 
    blood_sugar_fasting=85, blood_sugar_postmeal=120,
    weight_kg=65
)
print(f"\nLow Risk Input: BP=115/75, Fasting=85, Postmeal=120")
print(f"  → Prediction: {result['risk_level']} (confidence: {result['confidence']})")

# Medium risk profile
result = predict_physical_risk(
    bp_systolic=135, bp_diastolic=88,
    blood_sugar_fasting=110, blood_sugar_postmeal=165,
    weight_kg=78, edema=1, nausea_count=2
)
print(f"\nMedium Risk Input: BP=135/88, Fasting=110, Edema, Nausea")
print(f"  → Prediction: {result['risk_level']} (confidence: {result['confidence']})")

# High risk profile
result = predict_physical_risk(
    bp_systolic=160, bp_diastolic=105,
    blood_sugar_fasting=145, blood_sugar_postmeal=220,
    weight_kg=92, dizziness=1, edema=1, bleeding=1,
    pain_score=7, chronic_hypertension=1, gestational_diabetes=1
)
print(f"\nHigh Risk Input: BP=160/105, Fasting=145, Multiple Symptoms")
print(f"  → Prediction: {result['risk_level']} (confidence: {result['confidence']})")

---

## Summary

✅ **Physical Health Risk Model trained successfully!**

| Metric | Value |
|--------|-------|
| Algorithm | Ensemble (XGBoost / RF+LR) |
| Features | ~30 (including engineered) |
| Target | LOW / MEDIUM / HIGH |

### Key Risk Indicators
- **Hypertension**: BP ≥ 140/90 mmHg
- **Hyperglycemia**: Fasting ≥ 126 mg/dL or Postmeal ≥ 200 mg/dL
- **Symptoms**: Edema, Bleeding, Dizziness, Cramps

### Model Artifacts Saved
- `physical_health_ensemble.joblib` — Trained model
- `physical_health_scaler.joblib` — StandardScaler
- `physical_health_label_encoder.joblib` — LabelEncoder

---

⚠️ **Disclaimer**: This model predicts risk likelihood only — NOT a medical diagnosis.